# 16S and metabolomics CTF Analyses of the Longitudinal Acne Study

Date last updated: 1/7/2026

Notebook author: Britta De Pessemier, Yang Chen 

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

In [74]:
# Import essential Python packages
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import hashlib
from Bio import SeqIO

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.distance import permanova

# BIOM and Qiime2 related packages
import biom
import qiime2 as q2
from biom import load_table
from qiime2 import Artifact
import h5py

# Gemelli packages
from qiime2 import Artifact, Metadata
from qiime2.plugins import gemelli
from skbio.stats.ordination import OrdinationResults

# Stats
from skbio.stats.distance import DistanceMatrix
from skbio.stats.distance import permanova
from scipy.spatial.distance import pdist, squareform
from statsmodels.stats.multitest import multipletests
import itertools
from itertools import combinations

# Other
import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="You passed a edgecolor/edgecolors .*"
)


## Import table and metadata

In [75]:
biom_paths = {
    "16S_V1-V3": "../Data/16S/Tables/intersection_samples_16S_metaB/179426_feature-table_16S_V1V3_rare-11054_sampleIDfixed_16S-metaB-aligned.biom",
    "16S_V4": "../Data/16S/Tables/intersection_samples_16S_metaB/174951_feature-table_16S_V4_rare-3769_sampleIDfixed_16S-metaB-aligned.biom",
    "metabolomics": "../Data/metabolomics/Run3_10252024/metabolomics_method2_16S-metaB-aligned.biom"
}

metadata_path = "../Metadata/metadata_final_22102024.tsv"
metadata_df = pd.read_csv(metadata_path, sep="\t")
metadata_df = metadata_df.set_index("SampleID")
metadata_df.index.name = '#SampleID'

In [76]:
# Make qiime2 compatible metadata
qiime_md = Metadata(metadata_df)

## Plotting preparation

In [77]:
def draw_ellipse(mean, cov, ax, color, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor

    ell = Ellipse(
        xy=mean,
        width=width,
        height=height,
        angle=angle,
        edgecolor=color,
        facecolor=color,
        alpha=0.2,
        linewidth=0.5
    )
    ax.add_patch(ell)

In [78]:
group_colors = {
    "Healthy": "#000096",
    "Acne_L": "#ff676c",
    "Acne_NL": "#17c6d5"
}

group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

In [79]:
def build_tensor_robust(table, md, individual_id, state_column):
    """
    Build gemelli CTF tensor across gemelli versions.
    """

    b = build()

    if hasattr(b, "__call__"):
        return b(table, md, individual_id, state_column)

    if hasattr(b, "fit_transform"):
        return b.fit_transform(
            table=table,
            sample_metadata=md,
            individual_id_column=individual_id,
            state_column=state_column
        )

    if hasattr(b, "build"):
        return b.build(
            table,
            md,
            individual_id_column=individual_id,
            state_column=state_column
        )

    raise RuntimeError("Unsupported gemelli build API")


In [80]:
def calculate_pairwise_permanova(ordination_df, group_column='Group', n_permutations=999):
    """
    Calculate pairwise PERMANOVA between groups
    """
    # Create distance matrix from ordination coordinates
    coords = ordination_df[['PC1', 'PC2', 'PC3']].values
    dist_array = squareform(pdist(coords, metric='euclidean'))
    
    # Create DistanceMatrix object with sample IDs
    dm = DistanceMatrix(dist_array, ids=ordination_df.index.tolist())
    
    # Get unique groups
    groups = ordination_df[group_column].unique()
    group_pairs = list(combinations(groups, 2))
    
    # Calculate pairwise PERMANOVA
    pairwise_results = {}
    
    for group1, group2 in group_pairs:
        # Get sample IDs for the two groups
        ids1 = ordination_df[ordination_df[group_column] == group1].index.tolist()
        ids2 = ordination_df[ordination_df[group_column] == group2].index.tolist()
        pair_ids = ids1 + ids2
        
        # Filter distance matrix for these samples
        pair_dm = dm.filter(pair_ids)
        
        # Create grouping vector
        pair_grouping = [group1] * len(ids1) + [group2] * len(ids2)
        
        # Run PERMANOVA
        result = permanova(pair_dm, pair_grouping, permutations=n_permutations)
        
        pairwise_results[f"{group1}_vs_{group2}"] = {
            'statistic': result['test statistic'],
            'p_value': result['p-value'],
            'R2': result['test statistic'] / (result['number of groups'] - 1)  # pseudo R²
        }
    
    return pairwise_results

## Run Compositional Tensor Factorization (CTF) (C1 - left cheek)

In [81]:
ctf_results = {}

for region, biom_path in biom_paths.items():

    print(f"\n***Running CTF for {region}***")

    table_artifact = Artifact.import_data(
        "FeatureTable[Frequency]",
        biom_path
    )
    table_df = table_artifact.view(pd.DataFrame)

    shared_samples = table_df.index.intersection(metadata_df.index)
    if shared_samples.empty:
        raise ValueError(f"No shared samples between BIOM and metadata for {region}")

    table_df = table_df.loc[shared_samples]
    md_all = metadata_df.loc[shared_samples].copy()

    print(f"  Samples before C2 filter: {table_df.shape[0]}")
    print(f"  Unique subjects before C2 filter: {md_all['subject_ID'].nunique()}")

    keep_mask = md_all["c_zone"] != "C2"
    table_df = table_df.loc[keep_mask]
    md_all = md_all.loc[keep_mask]

    if table_df.empty:
        raise ValueError(f"All samples removed after C2 filtering for {region}")

    print(f"  Samples after C2 filter: {table_df.shape[0]}")
    print(f"  Unique subjects after C2 filter: {md_all['subject_ID'].nunique()}")

    subject_sample_counts = (
        md_all
        .groupby(["subject_ID_x_group", "group"])
        .size()
        .unstack(fill_value=0)
        .rename(columns={
            "Healthy": "H",
            "Acne_NL": "ANL",
            "Acne_L": "AL"
        })
    )

    for col in ["H", "ANL", "AL"]:
        if col not in subject_sample_counts.columns:
            subject_sample_counts[col] = 0

    subject_sample_counts["Total"] = subject_sample_counts[["H", "ANL", "AL"]].sum(axis=1)

    print("\n  Per-subject sample counts after C2 filter:")
    print(subject_sample_counts.sort_index())

    table_art = Artifact.import_data(
        "FeatureTable[Frequency]",
        table_df
    )

    md_all.index.name = "#SampleID"
    qiime_md = Metadata(md_all)

    ctf_res = gemelli.actions.ctf(
        table=table_art,
        sample_metadata=qiime_md,
        individual_id_column="subject_ID_x_group",
        state_column="sample_type",
        n_components=3
    )

    ordination = ctf_res.subject_biplot.view(OrdinationResults)

    spca_df = ordination.samples.iloc[:, :3].copy()
    spca_df.columns = ["PC1", "PC2", "PC3"]

    subj_md = (
        md_all
        .drop_duplicates("subject_ID_x_group")
        .set_index("subject_ID_x_group")
    )

    spca_df["Group"] = spca_df.index.map(subj_md["group"])
    spca_df["day"] = spca_df.index.map(subj_md["day"])
    spca_df["c_zone"] = spca_df.index.map(subj_md["c_zone"])

    subjects_retained = spca_df.index

    n_subjects_ctf = len(subjects_retained)
    n_samples_ctf = md_all.loc[
        md_all["subject_ID_x_group"].isin(subjects_retained)
    ].shape[0]

    # print("\n   Post-CTF summary:")
    # print(f"    Samples contributing: {n_samples_ctf}")
    # print(f"    Subjects retained: {n_subjects_ctf} --> should be this number of dots on plot")

    # Count subjects (dots) per group
    subject_group_counts = (
        subj_md
        .loc[subjects_retained]
        ["group"]
        .value_counts()
    )

    n_H = subject_group_counts.get("Healthy", 0)
    n_ANL = subject_group_counts.get("Acne_NL", 0)
    n_AL = subject_group_counts.get("Acne_L", 0)

    print("\n   Post-CTF summary:")
    print(f"       Samples contributing: {n_samples_ctf}")
    print(f"       Subjects retained: {n_subjects_ctf} --> should be this number of total dots on plot")
    print(f"          H subjects (dots): {n_H}")
    print(f"          ANL subjects (dots): {n_ANL}")
    print(f"          AL subjects (dots): {n_AL}")


    ctf_results[region] = {
        "ordination": ordination,
        "spca_df": spca_df,
        "metadata": subj_md
    }

    feature_df = ordination.features.iloc[:, :3].copy()
    feature_df.columns = ["PC1", "PC2", "PC3"]
    feature_df["feature_id"] = feature_df.index

    feature_df = (
        feature_df
        .assign(
            max_abs_loading=lambda x: x[["PC1", "PC2", "PC3"]].abs().max(axis=1)
        )
        .sort_values("max_abs_loading", ascending=False)
        .drop(columns="max_abs_loading")
    )

    outdir = "../Analyses/16S/CTF/feature_loadings"
    os.makedirs(outdir, exist_ok=True)

    feature_df.to_csv(
        f"{outdir}/CTF_feature_loadings_{region}_left-cheek.tsv",
        sep="\t",
        index=False
    )

    qza_outdir = "../Analyses/16S/CTF/subject_biplot_qza"
    os.makedirs(qza_outdir, exist_ok=True)

    subject_biplot_qza = (
        f"{qza_outdir}/CTF_subject_biplot_{region}_left-cheek.qza"
    )
    ctf_res.subject_biplot.save(subject_biplot_qza)

    print(f"  Saved subject biplot QZA: {subject_biplot_qza}")
    print('---------------------------------------------------------')



***Running CTF for 16S_V1-V3***
  Samples before C2 filter: 143
  Unique subjects before C2 filter: 17
  Samples after C2 filter: 87
  Unique subjects after C2 filter: 14

  Per-subject sample counts after C2 filter:
group               AL  ANL  H  Total
subject_ID_x_group                   
PP_18_Healthy        0    0  1      1
PP_1_Healthy         0    0  3      3
PP_304_Acne_L        6    0  0      6
PP_304_Acne_NL       0    3  0      3
PP_305_Acne_L        5    0  0      5
PP_305_Acne_NL       0    4  0      4
PP_306_Acne_L        4    0  0      4
PP_306_Acne_NL       0    2  0      2
PP_308_Acne_L        2    0  0      2
PP_308_Acne_NL       0    3  0      3
PP_310_Acne_L        5    0  0      5
PP_310_Acne_NL       0    4  0      4
PP_313_Acne_L        5    0  0      5
PP_313_Acne_NL       0    2  0      2
PP_314_Acne_L        5    0  0      5
PP_314_Acne_NL       0    3  0      3
PP_317_Acne_L        5    0  0      5
PP_317_Acne_NL       0    3  0      3
PP_318_Acne_L        5

/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_319_Acne_L,PP_317_Acne_NL,PP_308_Acne_NL,PP_317_Acne_L,PP_305_Acne_L,PP_306_Acne_NL,PP_314_Acne_L,PP_313_Acne_NL,PP_310_Acne_NL,PP_306_Acne_L,PP_305_Acne_NL,PP_304_Acne_L,PP_308_Acne_L,PP_318_Acne_L,PP_319_Acne_NL,PP_313_Acne_L,PP_310_Acne_L,PP_1_Healthy,PP_314_Acne_NL,PP_318_Acne_NL,PP_304_Acne_NL) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),



   Post-CTF summary:
       Samples contributing: 87
       Subjects retained: 24 --> should be this number of total dots on plot
          H subjects (dots): 4
          ANL subjects (dots): 10
          AL subjects (dots): 10
  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_16S_V1-V3_left-cheek.qza
---------------------------------------------------------

***Running CTF for 16S_V4***
  Samples before C2 filter: 143
  Unique subjects before C2 filter: 17
  Samples after C2 filter: 87
  Unique subjects after C2 filter: 14

  Per-subject sample counts after C2 filter:
group               AL  ANL  H  Total
subject_ID_x_group                   
PP_18_Healthy        0    0  1      1
PP_1_Healthy         0    0  3      3
PP_304_Acne_L        6    0  0      6
PP_304_Acne_NL       0    3  0      3
PP_305_Acne_L        5    0  0      5
PP_305_Acne_NL       0    4  0      4
PP_306_Acne_L        4    0  0      4
PP_306_Acne_NL       0    2  0      2
PP_308_

/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_319_Acne_L,PP_317_Acne_NL,PP_308_Acne_NL,PP_317_Acne_L,PP_305_Acne_L,PP_306_Acne_NL,PP_314_Acne_L,PP_313_Acne_NL,PP_310_Acne_NL,PP_306_Acne_L,PP_305_Acne_NL,PP_304_Acne_L,PP_308_Acne_L,PP_318_Acne_L,PP_319_Acne_NL,PP_313_Acne_L,PP_310_Acne_L,PP_1_Healthy,PP_314_Acne_NL,PP_318_Acne_NL,PP_304_Acne_NL) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),



   Post-CTF summary:
       Samples contributing: 87
       Subjects retained: 24 --> should be this number of total dots on plot
          H subjects (dots): 4
          ANL subjects (dots): 10
          AL subjects (dots): 10
  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_16S_V4_left-cheek.qza
---------------------------------------------------------

***Running CTF for metabolomics***
  Samples before C2 filter: 143
  Unique subjects before C2 filter: 17
  Samples after C2 filter: 87
  Unique subjects after C2 filter: 14

  Per-subject sample counts after C2 filter:
group               AL  ANL  H  Total
subject_ID_x_group                   
PP_18_Healthy        0    0  1      1
PP_1_Healthy         0    0  3      3
PP_304_Acne_L        6    0  0      6
PP_304_Acne_NL       0    3  0      3
PP_305_Acne_L        5    0  0      5
PP_305_Acne_NL       0    4  0      4
PP_306_Acne_L        4    0  0      4
PP_306_Acne_NL       0    2  0      2
PP_3

/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_319_Acne_L,PP_317_Acne_NL,PP_308_Acne_NL,PP_317_Acne_L,PP_305_Acne_L,PP_306_Acne_NL,PP_314_Acne_L,PP_313_Acne_NL,PP_310_Acne_NL,PP_306_Acne_L,PP_305_Acne_NL,PP_304_Acne_L,PP_308_Acne_L,PP_318_Acne_L,PP_319_Acne_NL,PP_313_Acne_L,PP_310_Acne_L,PP_1_Healthy,PP_314_Acne_NL,PP_318_Acne_NL,PP_304_Acne_NL) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),



   Post-CTF summary:
       Samples contributing: 87
       Subjects retained: 24 --> should be this number of total dots on plot
          H subjects (dots): 4
          ANL subjects (dots): 10
          AL subjects (dots): 10
  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_metabolomics_left-cheek.qza
---------------------------------------------------------


In [82]:
np.random.seed(123)

# Manual layout presets per plot
LAYOUT_PRESETS = {
    "16S_V1-V3": {
        "group_legend_xy": (0.63, 0.0006),
        "permanova_xy": (0.643, 0.98),
        "legend_loc": "lower left"
    },
    "16S_V4": {
        "group_legend_xy": (0.63, 0.0006),
        "permanova_xy": (0.643, 0.98),
        "legend_loc": "lower left"
    },
    "metabolomics": {
        "group_legend_xy": (0.63, 0.0006),
        "permanova_xy": (0.02, 0.98),
        "legend_loc": "lower left"
    }
}

# Group name abbreviations for legend and annotations
group_abbrev = {
    'Healthy': 'H',
    'Acne_L': 'AL',
    'Acne_NL': 'ANL'
}

for region, res in ctf_results.items():

    ordination = res["ordination"]
    spca_df = res["spca_df"]

    # Select layout preset for this plot
    if region not in LAYOUT_PRESETS:
        raise ValueError(f"No layout preset defined for region: {region}")
    layout = LAYOUT_PRESETS[region]

    # Run pairwise PERMANOVA
    permanova_results = calculate_pairwise_permanova(
        spca_df,
        group_column="Group"
    )

    # FDR correction
    comparisons = list(permanova_results.keys())
    p_values = [v["p_value"] for v in permanova_results.values()]
    _, qvals, _, _ = multipletests(p_values, method="fdr_bh")

    for i, comp in enumerate(comparisons):
        permanova_results[comp]["adjusted_p_value"] = qvals[i]

    # Axis variance explained
    pc1_var = ordination.proportion_explained[0] * 100
    pc2_var = ordination.proportion_explained[1] * 100

    fig, ax = plt.subplots(figsize=(8, 8))

    # Scatter points and ellipses
    plot_df = spca_df.copy()

    for group, df_g in plot_df.groupby("Group"):

        color = group_colors[group]
        n_subjects = df_g.shape[0]
        label = f"{group_abbrev[group]} (n={n_subjects} subj. over time)"

        ax.scatter(
            df_g["PC1"],
            df_g["PC2"],
            s=120,
            color=color,
            edgecolor="black",
            linewidth=0.6,
            alpha=1,
            label=label
        )

        if len(df_g) > 2:
            pts = df_g[["PC1", "PC2"]].values
            draw_ellipse(
                mean=pts.mean(axis=0),
                cov=np.cov(pts, rowvar=False),
                ax=ax,
                color=color
            )

    # Axis labels and title
    ax.set_xlabel(f"PC1 ({pc1_var:.2f}%)", fontsize=16)
    ax.set_ylabel(f"PC2 ({pc2_var:.2f}%)", fontsize=16)
    title_region = region.replace("16S_", "16S ")
    ax.set_title(f"CTF {title_region} (left cheek)", fontsize=18)

    # Group legend with manual position per modality
    legend = ax.legend(
        frameon=True,
        fancybox=True,
        bbox_to_anchor=layout["group_legend_xy"],
        loc=layout["legend_loc"]
    )

    # Explicit legend frame styling
    frame = legend.get_frame()
    frame.set_facecolor("white")
    frame.set_edgecolor("grey")
    frame.set_linewidth(1.0)
    frame.set_alpha(0.9)

    legend.get_frame().set_linewidth(1.0)

    # Build PERMANOVA annotation text
    annotation_lines = []
    for comp, stats in permanova_results.items():

        g1, g2 = comp.split("_vs_")
        g1 = group_abbrev.get(g1, g1)
        g2 = group_abbrev.get(g2, g2)

        q = stats["adjusted_p_value"]

        if q < 0.001:
            ptxt = f"*** p={q:.3f}"
        elif q < 0.01:
            ptxt = f"** p={q:.3f}"
        elif q < 0.05:
            ptxt = f"* p={q:.3f}"
        else:
            ptxt = f"p={q:.3f}"

        annotation_lines.append(
            f"{g1} vs {g2}: {ptxt}, F={stats['statistic']:.3f}"
        )

    full_annotation = (
        "PERMANOVA (FDR-adjusted):\n" +
        "\n".join(annotation_lines)
    )

    # PERMANOVA annotation box with manual position per modality
    ax.text(
        layout["permanova_xy"][0],
        layout["permanova_xy"][1],
        full_annotation,
        transform=ax.transAxes,
        fontsize=11,
        verticalalignment="top",
        horizontalalignment="left",
        bbox=dict(
            boxstyle="round,pad=0.5",
            facecolor="white",
            edgecolor="grey",
            linewidth=1,
            alpha=0.75
        )
    )

    # Save figure
    plt.tight_layout()
    plt.savefig(
        f"../Figures/Main/Figure_2/CTF_{region}_left-cheek.png",
        dpi=600,
        bbox_inches="tight"
    )
    plt.close()

    # Print PERMANOVA results to console
    print(f"\n=== PERMANOVA results for {region} ===")
    for comp, stats in permanova_results.items():
        print(f"{comp}:")
        print(f"  Raw p-value: {stats['p_value']:.3f}")
        print(f"  FDR-adjusted q-value: {stats['adjusted_p_value']:.3f}")
        print(f"  F-statistic: {stats['statistic']:.3f}")



=== PERMANOVA results for 16S_V1-V3 ===
Healthy_vs_Acne_L:
  Raw p-value: 0.009
  FDR-adjusted q-value: 0.027
  F-statistic: 3.207
Healthy_vs_Acne_NL:
  Raw p-value: 0.155
  FDR-adjusted q-value: 0.217
  F-statistic: 1.815
Acne_L_vs_Acne_NL:
  Raw p-value: 0.217
  FDR-adjusted q-value: 0.217
  F-statistic: 1.537

=== PERMANOVA results for 16S_V4 ===
Healthy_vs_Acne_NL:
  Raw p-value: 0.015
  FDR-adjusted q-value: 0.022
  F-statistic: 3.639
Healthy_vs_Acne_L:
  Raw p-value: 0.001
  FDR-adjusted q-value: 0.003
  F-statistic: 5.182
Acne_NL_vs_Acne_L:
  Raw p-value: 0.154
  FDR-adjusted q-value: 0.154
  F-statistic: 1.869

=== PERMANOVA results for metabolomics ===
Healthy_vs_Acne_NL:
  Raw p-value: 0.077
  FDR-adjusted q-value: 0.116
  F-statistic: 2.401
Healthy_vs_Acne_L:
  Raw p-value: 0.009
  FDR-adjusted q-value: 0.027
  F-statistic: 4.589
Acne_NL_vs_Acne_L:
  Raw p-value: 0.311
  FDR-adjusted q-value: 0.311
  F-statistic: 1.174


## Run Compositional Tensor Factorization (CTF) (C2 - right cheek)

In [83]:
ctf_results = {}

for region, biom_path in biom_paths.items():

    print(f"\n***Running CTF for {region}***")

    table_artifact = Artifact.import_data(
        "FeatureTable[Frequency]",
        biom_path
    )
    table_df = table_artifact.view(pd.DataFrame)

    shared_samples = table_df.index.intersection(metadata_df.index)
    if shared_samples.empty:
        raise ValueError(f"No shared samples between BIOM and metadata for {region}")

    table_df = table_df.loc[shared_samples]
    md_all = metadata_df.loc[shared_samples].copy()

    print(f"  Samples before C1 filter: {table_df.shape[0]}")
    print(f"  Unique subjects before C1 filter: {md_all['subject_ID'].nunique()}")

    keep_mask = md_all["c_zone"] != "C1"
    table_df = table_df.loc[keep_mask]
    md_all = md_all.loc[keep_mask]

    if table_df.empty:
        raise ValueError(f"All samples removed after C1 filtering for {region}")

    print(f"  Samples after C1 filter: {table_df.shape[0]}")
    print(f"  Unique subjects after C1 filter: {md_all['subject_ID'].nunique()}")

    subject_sample_counts = (
        md_all
        .groupby(["subject_ID_x_group", "group"])
        .size()
        .unstack(fill_value=0)
        .rename(columns={
            "Healthy": "H",
            "Acne_NL": "ANL",
            "Acne_L": "AL"
        })
    )

    for col in ["H", "ANL", "AL"]:
        if col not in subject_sample_counts.columns:
            subject_sample_counts[col] = 0

    subject_sample_counts["Total"] = subject_sample_counts[["H", "ANL", "AL"]].sum(axis=1)

    print("\n  Per-subject sample counts after C1 filter:")
    print(subject_sample_counts.sort_index())

    table_art = Artifact.import_data(
        "FeatureTable[Frequency]",
        table_df
    )

    md_all.index.name = "#SampleID"
    qiime_md = Metadata(md_all)

    ctf_res = gemelli.actions.ctf(
        table=table_art,
        sample_metadata=qiime_md,
        individual_id_column="subject_ID_x_group",
        state_column="sample_type",
        n_components=3
    )

    ordination = ctf_res.subject_biplot.view(OrdinationResults)

    spca_df = ordination.samples.iloc[:, :3].copy()
    spca_df.columns = ["PC1", "PC2", "PC3"]

    subj_md = (
        md_all
        .drop_duplicates("subject_ID_x_group")
        .set_index("subject_ID_x_group")
    )

    spca_df["Group"] = spca_df.index.map(subj_md["group"])
    spca_df["day"] = spca_df.index.map(subj_md["day"])
    spca_df["c_zone"] = spca_df.index.map(subj_md["c_zone"])

    subjects_retained = spca_df.index

    n_subjects_ctf = len(subjects_retained)
    n_samples_ctf = md_all.loc[
        md_all["subject_ID_x_group"].isin(subjects_retained)
    ].shape[0]

    subject_group_counts = (
        subj_md
        .loc[subjects_retained]
        ["group"]
        .value_counts()
    )

    n_H = subject_group_counts.get("Healthy", 0)
    n_ANL = subject_group_counts.get("Acne_NL", 0)
    n_AL = subject_group_counts.get("Acne_L", 0)

    print("\n   Post-CTF summary:")
    print(f"       Samples contributing: {n_samples_ctf}")
    print(f"       Subjects retained: {n_subjects_ctf} --> should be this number of total dots on plot")
    print(f"          H subjects (dots): {n_H}")
    print(f"          ANL subjects (dots): {n_ANL}")
    print(f"          AL subjects (dots): {n_AL}")

    ctf_results[region] = {
        "ordination": ordination,
        "spca_df": spca_df,
        "metadata": subj_md
    }

    feature_df = ordination.features.iloc[:, :3].copy()
    feature_df.columns = ["PC1", "PC2", "PC3"]
    feature_df["feature_id"] = feature_df.index

    feature_df = (
        feature_df
        .assign(
            max_abs_loading=lambda x: x[["PC1", "PC2", "PC3"]].abs().max(axis=1)
        )
        .sort_values("max_abs_loading", ascending=False)
        .drop(columns="max_abs_loading")
    )

    outdir = "../Analyses/16S/CTF/feature_loadings"
    os.makedirs(outdir, exist_ok=True)

    feature_df.to_csv(
        f"{outdir}/CTF_feature_loadings_{region}_right-cheek.tsv",
        sep="\t",
        index=False
    )

    qza_outdir = "../Analyses/16S/CTF/subject_biplot_qza"
    os.makedirs(qza_outdir, exist_ok=True)

    subject_biplot_qza = (
        f"{qza_outdir}/CTF_subject_biplot_{region}_right-cheek.qza"
    )
    ctf_res.subject_biplot.save(subject_biplot_qza)

    print(f"  Saved subject biplot QZA: {subject_biplot_qza}")
    print('---------------------------------------------------------')



***Running CTF for 16S_V1-V3***
  Samples before C1 filter: 143
  Unique subjects before C1 filter: 17
  Samples after C1 filter: 89
  Unique subjects after C1 filter: 15

  Per-subject sample counts after C1 filter:
group               AL  ANL  H  Total
subject_ID_x_group                   
PP_11_Healthy        0    0  1      1
PP_17_Healthy        0    0  2      2
PP_18_Healthy        0    0  1      1
PP_304_Acne_L        6    0  0      6
PP_304_Acne_NL       0    3  0      3
PP_305_Acne_L        5    0  0      5
PP_305_Acne_NL       0    4  0      4
PP_306_Acne_L        3    0  0      3
PP_306_Acne_NL       0    2  0      2
PP_308_Acne_L        4    0  0      4
PP_308_Acne_NL       0    3  0      3
PP_310_Acne_L        3    0  0      3
PP_310_Acne_NL       0    4  0      4
PP_313_Acne_L        5    0  0      5
PP_313_Acne_NL       0    2  0      2
PP_314_Acne_L        5    0  0      5
PP_314_Acne_NL       0    3  0      3
PP_317_Acne_L        7    0  0      7
PP_317_Acne_NL       0

/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_319_Acne_L,PP_317_Acne_NL,PP_308_Acne_NL,PP_317_Acne_L,PP_305_Acne_L,PP_306_Acne_NL,PP_314_Acne_L,PP_313_Acne_NL,PP_310_Acne_NL,PP_17_Healthy,PP_306_Acne_L,PP_305_Acne_NL,PP_304_Acne_L,PP_308_Acne_L,PP_318_Acne_L,PP_319_Acne_NL,PP_313_Acne_L,PP_310_Acne_L,PP_314_Acne_NL,PP_318_Acne_NL,PP_304_Acne_NL) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),



   Post-CTF summary:
       Samples contributing: 89
       Subjects retained: 25 --> should be this number of total dots on plot
          H subjects (dots): 5
          ANL subjects (dots): 10
          AL subjects (dots): 10
  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_16S_V1-V3_right-cheek.qza
---------------------------------------------------------

***Running CTF for 16S_V4***
  Samples before C1 filter: 143
  Unique subjects before C1 filter: 17
  Samples after C1 filter: 89
  Unique subjects after C1 filter: 15

  Per-subject sample counts after C1 filter:
group               AL  ANL  H  Total
subject_ID_x_group                   
PP_11_Healthy        0    0  1      1
PP_17_Healthy        0    0  2      2
PP_18_Healthy        0    0  1      1
PP_304_Acne_L        6    0  0      6
PP_304_Acne_NL       0    3  0      3
PP_305_Acne_L        5    0  0      5
PP_305_Acne_NL       0    4  0      4
PP_306_Acne_L        3    0  0      3
PP_306

/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_319_Acne_L,PP_317_Acne_NL,PP_308_Acne_NL,PP_317_Acne_L,PP_305_Acne_L,PP_306_Acne_NL,PP_314_Acne_L,PP_313_Acne_NL,PP_310_Acne_NL,PP_17_Healthy,PP_306_Acne_L,PP_305_Acne_NL,PP_304_Acne_L,PP_308_Acne_L,PP_318_Acne_L,PP_319_Acne_NL,PP_313_Acne_L,PP_310_Acne_L,PP_314_Acne_NL,PP_318_Acne_NL,PP_304_Acne_NL) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),



   Post-CTF summary:
       Samples contributing: 89
       Subjects retained: 25 --> should be this number of total dots on plot
          H subjects (dots): 5
          ANL subjects (dots): 10
          AL subjects (dots): 10
  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_16S_V4_right-cheek.qza
---------------------------------------------------------

***Running CTF for metabolomics***
  Samples before C1 filter: 143
  Unique subjects before C1 filter: 17
  Samples after C1 filter: 89
  Unique subjects after C1 filter: 15

  Per-subject sample counts after C1 filter:
group               AL  ANL  H  Total
subject_ID_x_group                   
PP_11_Healthy        0    0  1      1
PP_17_Healthy        0    0  2      2
PP_18_Healthy        0    0  1      1
PP_304_Acne_L        6    0  0      6
PP_304_Acne_NL       0    3  0      3
PP_305_Acne_L        5    0  0      5
PP_305_Acne_NL       0    4  0      4
PP_306_Acne_L        3    0  0      3
PP_

/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_319_Acne_L,PP_317_Acne_NL,PP_308_Acne_NL,PP_317_Acne_L,PP_305_Acne_L,PP_306_Acne_NL,PP_314_Acne_L,PP_313_Acne_NL,PP_310_Acne_NL,PP_17_Healthy,PP_306_Acne_L,PP_305_Acne_NL,PP_304_Acne_L,PP_308_Acne_L,PP_318_Acne_L,PP_319_Acne_NL,PP_313_Acne_L,PP_310_Acne_L,PP_314_Acne_NL,PP_318_Acne_NL,PP_304_Acne_NL) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),



   Post-CTF summary:
       Samples contributing: 89
       Subjects retained: 25 --> should be this number of total dots on plot
          H subjects (dots): 5
          ANL subjects (dots): 10
          AL subjects (dots): 10
  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_metabolomics_right-cheek.qza
---------------------------------------------------------


In [84]:
np.random.seed(69)

# Layout presets per modality (reuse same dict as left cheek)
LAYOUT_PRESETS = {
    "16S_V1-V3": {
        "group_legend_xy": (0.63, 0.0006),
        "permanova_xy": (0.632, 0.98),
        "legend_loc": "lower left"
    },
    "16S_V4": {
        "group_legend_xy": (0.63, 0.0006),
        "permanova_xy": (0.643, 0.98),
        "legend_loc": "lower left"
    },
    "metabolomics": {
        "group_legend_xy": (0.63, 0.0006),
        "permanova_xy": (0.02, 0.98),
        "legend_loc": "lower left"
    }
}

group_abbrev = {
    'Healthy': 'H',
    'Acne_L': 'AL',
    'Acne_NL': 'ANL'
}

for region, res in ctf_results.items():

    ordination = res["ordination"]
    spca_df = res["spca_df"]

    if region not in LAYOUT_PRESETS:
        raise ValueError(f"No layout preset defined for region: {region}")
    layout = LAYOUT_PRESETS[region]

    # PERMANOVA
    permanova_results = calculate_pairwise_permanova(
        spca_df, group_column="Group"
    )

    comparisons = list(permanova_results.keys())
    p_values = [v["p_value"] for v in permanova_results.values()]
    _, qvals, _, _ = multipletests(p_values, method="fdr_bh")

    for i, comp in enumerate(comparisons):
        permanova_results[comp]["adjusted_p_value"] = qvals[i]

    pc1_var = ordination.proportion_explained[0] * 100
    pc2_var = ordination.proportion_explained[1] * 100

    fig, ax = plt.subplots(figsize=(8, 8))

    # Scatter + ellipses
    for group, df_g in spca_df.groupby("Group"):

        color = group_colors[group]
        n_samples = df_g.shape[0]
        label = f"{group_abbrev[group]} (n={n_samples} subj. over time)"

        ax.scatter(
            df_g["PC1"],
            df_g["PC2"],
            s=120,
            color=color,
            edgecolor="black",
            linewidth=0.6,
            alpha=1,
            label=label
        )

        if len(df_g) > 2:
            pts = df_g[["PC1", "PC2"]].values
            draw_ellipse(
                mean=pts.mean(axis=0),
                cov=np.cov(pts, rowvar=False),
                ax=ax,
                color=color
            )

    # Labels & title
    ax.set_xlabel(f"PC1 ({pc1_var:.2f}%)", fontsize=16)
    ax.set_ylabel(f"PC2 ({pc2_var:.2f}%)", fontsize=16)

    title_region = region.replace("16S_", "16S ")
    ax.set_title(f"CTF {title_region} (right cheek)", fontsize=18)

    # Legend (layout-controlled)
    legend = ax.legend(
        frameon=True,
        fancybox=True,
        bbox_to_anchor=layout["group_legend_xy"],
        loc=layout["legend_loc"]
    )

    frame = legend.get_frame()
    frame.set_facecolor("white")
    frame.set_edgecolor("grey")
    frame.set_linewidth(1.0)
    frame.set_alpha(0.9)

    # PERMANOVA annotation
    annotation_lines = []
    for comp, stats in permanova_results.items():

        g1, g2 = comp.split("_vs_")
        g1 = group_abbrev.get(g1, g1)
        g2 = group_abbrev.get(g2, g2)

        q = stats["adjusted_p_value"]

        if q < 0.001:
            ptxt = f"*** p={q:.3f}"
        elif q < 0.01:
            ptxt = f"** p={q:.3f}"
        elif q < 0.05:
            ptxt = f"* p={q:.3f}"
        else:
            ptxt = f"p={q:.3f}"

        annotation_lines.append(
            f"{g1} vs {g2}: {ptxt}, F={stats['statistic']:.3f}"
        )

    full_annotation = (
        "PERMANOVA (FDR-adjusted):\n" +
        "\n".join(annotation_lines)
    )

    ax.text(
        layout["permanova_xy"][0],
        layout["permanova_xy"][1],
        full_annotation,
        transform=ax.transAxes,
        fontsize=11,
        verticalalignment="top",
        horizontalalignment="left",
        bbox=dict(
            boxstyle="round,pad=0.5",
            facecolor="white",
            edgecolor="grey",
            linewidth=1,
            alpha=0.75
        )
    )

    # Save
    plt.tight_layout()
    plt.savefig(
        f"../Figures/Main/Figure_2/CTF_{region}_right-cheek.png",
        dpi=600,
        bbox_inches="tight"
    )
    plt.close()

    # Console output
    print(f"\n=== PERMANOVA results for {region} (right cheek) ===")
    for comp, stats in permanova_results.items():
        print(f"{comp}:")
        print(f"  Raw p-value: {stats['p_value']:.3f}")
        print(f"  FDR-adjusted q-value: {stats['adjusted_p_value']:.3f}")
        print(f"  F-statistic: {stats['statistic']:.3f}")



=== PERMANOVA results for 16S_V1-V3 (right cheek) ===
Healthy_vs_Acne_NL:
  Raw p-value: 0.172
  FDR-adjusted q-value: 0.172
  F-statistic: 1.712
Healthy_vs_Acne_L:
  Raw p-value: 0.004
  FDR-adjusted q-value: 0.012
  F-statistic: 5.303
Acne_NL_vs_Acne_L:
  Raw p-value: 0.138
  FDR-adjusted q-value: 0.172
  F-statistic: 2.048

=== PERMANOVA results for 16S_V4 (right cheek) ===
Healthy_vs_Acne_NL:
  Raw p-value: 0.095
  FDR-adjusted q-value: 0.143
  F-statistic: 2.435
Healthy_vs_Acne_L:
  Raw p-value: 0.004
  FDR-adjusted q-value: 0.012
  F-statistic: 4.212
Acne_NL_vs_Acne_L:
  Raw p-value: 0.147
  FDR-adjusted q-value: 0.147
  F-statistic: 1.816

=== PERMANOVA results for metabolomics (right cheek) ===
Healthy_vs_Acne_NL:
  Raw p-value: 0.312
  FDR-adjusted q-value: 0.312
  F-statistic: 1.293
Healthy_vs_Acne_L:
  Raw p-value: 0.014
  FDR-adjusted q-value: 0.042
  F-statistic: 3.674
Acne_NL_vs_Acne_L:
  Raw p-value: 0.196
  FDR-adjusted q-value: 0.294
  F-statistic: 1.669


## Map taxonomy of each ASV feature loading within each CTF model

In [85]:
ctf_dir = "../Analyses/16S/CTF/feature_loadings"

primer_sets = ["V1-V3", "V4"]
cheek_sides = ["left", "right"]

for primer_set in primer_sets:
    for cheek_side in cheek_sides:

        ctf_path = (
            f"{ctf_dir}/CTF_feature_loadings_16S_"
            f"{primer_set}_{cheek_side}-cheek.tsv"
        )

        if not os.path.exists(ctf_path):
            print(f"Skipping missing file: {ctf_path}")
            continue

        print(f"Annotating {os.path.basename(ctf_path)}")

        df = pd.read_csv(ctf_path, sep="\t")

        if "feature_id" not in df.columns:
            raise ValueError(f"'feature_id' column missing in {ctf_path}")

        # Select FASTA based on primer set
        if primer_set == "V1-V3":
            fasta_file = "../Data/16S/Fasta/179426_V1V3_ASVs.fasta"
        elif primer_set == "V4":
            fasta_file = "../Data/16S/Fasta/174951_V4_ASVs.fasta"
        else:
            raise ValueError(f"Unknown primer set: {primer_set}")

        # Build sequence → ASV-hash mapping from FASTA
        seq_to_hash = {}

        for record in SeqIO.parse(fasta_file, "fasta"):
            seq = str(record.seq).upper()
            asv_hash = record.id
            seq_to_hash[seq] = asv_hash

        # Map feature_id (sequence) → asv_hash
        df["asv_hash"] = df["feature_id"].str.upper().map(seq_to_hash)

        # Sanity check
        n_missing = df["asv_hash"].isna().sum()
        if n_missing > 0:
            raise ValueError(
                f"{n_missing} feature_id sequences in {ctf_path} "
                f"were not found in {os.path.basename(fasta_file)}"
            )

        # Write back to disk
        df.to_csv(ctf_path, sep="\t", index=False)


Annotating CTF_feature_loadings_16S_V1-V3_left-cheek.tsv
Annotating CTF_feature_loadings_16S_V1-V3_right-cheek.tsv
Annotating CTF_feature_loadings_16S_V4_left-cheek.tsv
Annotating CTF_feature_loadings_16S_V4_right-cheek.tsv


In [86]:
# Paths and configuration
ctf_dir = "../Analyses/16S/CTF/feature_loadings"

primer_sets = ["V1-V3", "V4"]
cheek_sides = ["left", "right"]

taxonomy_paths = {
    "V1-V3": "../Analyses/16S/SEPP/179426_V1V3_GG2_taxonomy/taxonomy.tsv",
    "V4": "../Analyses/16S/SEPP/174951_V4_GG2_taxonomy/taxonomy.tsv",
}

def extract_lowest_taxon(taxon):
    if pd.isna(taxon):
        return "Unclassified"

    parts = [p.strip() for p in taxon.split(";") if p.strip()]

    # Preferred rank order (lowest → highest)
    rank_prefixes = [
        "g__",  # genus
        "f__",  # family
        "o__",  # order
        "c__",  # class
        "p__",  # phylum
        "d__",  # domain
    ]

    for prefix in rank_prefixes:
        for p in parts:
            if p.startswith(prefix) and p != prefix:
                return p

    return "Unclassified"


# Load taxonomy mappings (asv_hash → Taxon)
tax_maps = {}

for primer_set, tax_path in taxonomy_paths.items():
    if not os.path.exists(tax_path):
        raise FileNotFoundError(f"Missing taxonomy file: {tax_path}")

    tax_df = pd.read_csv(tax_path, sep="\t")

    required_cols = {"Feature ID", "Taxon"}
    if not required_cols.issubset(tax_df.columns):
        raise ValueError(f"taxonomy.tsv malformed: {tax_path}")

    tax_maps[primer_set] = dict(
        zip(tax_df["Feature ID"], tax_df["Taxon"])
    )

# Annotate CTF feature loadings
for primer_set in primer_sets:
    for cheek_side in cheek_sides:

        ctf_path = (
            f"{ctf_dir}/CTF_feature_loadings_16S_"
            f"{primer_set}_{cheek_side}-cheek.tsv"
        )

        if not os.path.exists(ctf_path):
            print(f"Skipping missing file: {ctf_path}")
            continue

        print(f"Annotating taxonomy: {os.path.basename(ctf_path)}")

        df = pd.read_csv(ctf_path, sep="\t")

        if "asv_hash" not in df.columns:
            raise ValueError(f"'asv_hash' column missing in {ctf_path}")

        # Map full taxonomy
        df["Taxon"] = df["asv_hash"].map(tax_maps[primer_set])

        # Sanity check
        missing = df["Taxon"].isna().sum()
        if missing > 0:
            raise ValueError(
                f"{missing} ASVs in {ctf_path} "
                f"were not found in {primer_set} taxonomy.tsv"
            )

        # Add lowest resolved taxon
        df["lowest_taxon"] = df["Taxon"].apply(extract_lowest_taxon)

        # Write annotated file
        df.to_csv(ctf_path, sep="\t", index=False)


Annotating taxonomy: CTF_feature_loadings_16S_V1-V3_left-cheek.tsv
Annotating taxonomy: CTF_feature_loadings_16S_V1-V3_right-cheek.tsv
Annotating taxonomy: CTF_feature_loadings_16S_V4_left-cheek.tsv
Annotating taxonomy: CTF_feature_loadings_16S_V4_right-cheek.tsv


## Plot feature importances

In [87]:
# Paths
CTF_DIR = "../Analyses/16S/CTF/feature_loadings"
FIG_OUT_DIR = "../Figures/Supplementary/Suppl_Figure_2/"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

# Parameters
TOP_N = 10
PRIMERS = ["V1-V3", "V4"]
CHEEKS = ["left", "right"]

# Per-cheek CTF feature importance
for primer in PRIMERS:
    for cheek in CHEEKS:

        tsv_path = f"{CTF_DIR}/CTF_feature_loadings_16S_{primer}_{cheek}-cheek.tsv"

        if not os.path.exists(tsv_path):
            print(f"Skipping missing file: {tsv_path}")
            continue

        print(f"\nProcessing 16S {primer} {cheek} cheek")

        df = pd.read_csv(tsv_path, sep="\t")

        required_cols = {"feature_id", "PC1", "PC2", "PC3", "asv_hash", "Taxon", "lowest_taxon"}
        if not required_cols.issubset(df.columns):
            raise ValueError(f"Missing required columns in {tsv_path}")

        # CTF loading magnitude
        df["importance"] = np.sqrt(
            df["PC1"]**2 + df["PC2"]**2 + df["PC3"]**2
        )

        # Rank features
        df = df.sort_values("importance", ascending=False)

        # Overwrite TSV with importance + labels
        df.to_csv(tsv_path, sep="\t", index=False)

        # Plot top N features
        plot_df = df.head(TOP_N).iloc[::-1]
        y_pos = np.arange(len(plot_df))

        fig, ax = plt.subplots(figsize=(6, 0.35 * TOP_N + 1))
        ax.barh(y_pos, plot_df["importance"])
        ax.set_yticks(y_pos)
        ax.set_yticklabels(plot_df["lowest_taxon"] + " ASV")

        ax.set_xlabel("CTF loading magnitude (√(PC1² + PC2² + PC3²))")
        ax.set_title(f"Top {TOP_N} CTF features\n16S {primer} {cheek} cheek")

        plt.tight_layout()

        out_png = os.path.join(
            FIG_OUT_DIR,
            f"16S_{primer}_{cheek}-cheek_top{TOP_N}_CTF_features.png"
        )
        plt.savefig(out_png, dpi=600)
        plt.close()

        print(f"Saved {out_png}")

print("\nAll per-cheek CTF feature importance plots generated.")



Processing 16S V1-V3 left cheek
Saved ../Figures/Supplementary/Suppl_Figure_2/16S_V1-V3_left-cheek_top10_CTF_features.png

Processing 16S V1-V3 right cheek
Saved ../Figures/Supplementary/Suppl_Figure_2/16S_V1-V3_right-cheek_top10_CTF_features.png

Processing 16S V4 left cheek
Saved ../Figures/Supplementary/Suppl_Figure_2/16S_V4_left-cheek_top10_CTF_features.png

Processing 16S V4 right cheek
Saved ../Figures/Supplementary/Suppl_Figure_2/16S_V4_right-cheek_top10_CTF_features.png

All per-cheek CTF feature importance plots generated.
